In [1]:
import os
from pathlib import Path
from typing import List, Dict, Tuple

INPUT_DIR = Path("/kaggle/input/datasets/neron1/text2summary/form10_text")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_FILE = OUTPUT_DIR / "summaries.csv"
            
MODEL_REPO = "bartowski/Qwen2.5-14B-Instruct-GGUF"
MODEL_FILE = "Qwen2.5-14B-Instruct-Q6_K.gguf"
MODEL_CACHE = Path("/kaggle/working/models")
MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"

MAX_CONTEXT = 32768
TEMPERATURE = 0.1
MAX_TOKENS = 2048

In [2]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 GB 497.9 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00


In [5]:
!apt-get update && apt-get install -y aria2

download_url = f"https://huggingface.co/{MODEL_REPO}/resolve/main/{MODEL_FILE}"
save_path = f"{MODEL_CACHE}"

!aria2c -x 16 -s 16 -c -d {save_path} -o {MODEL_FILE} "{download_url}"

model_path = os.path.join(save_path, MODEL_FILE)
if os.path.exists(model_path):
    print(f"File size: {os.path.getsize(model_path) / (1024**3):.2f} GB")
else:
    print("Download failed")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,497 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [354 B]      
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,9

In [6]:
from llama_cpp import Llama

print("Loading model to GPU")

llm = Llama(
    model_path=model_path,
    n_ctx=16384,                  
    n_gpu_layers=-1,             
    n_batch=512,                 
    n_threads=4,                 
    verbose=False                
)

print("\nModel successfully loaded on GPU")

Loading model to GPU...


ggml_cuda_init: found 2 CUDA devices (Total VRAM: 29825 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB
  Device 1: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB
llama_model_load_from_file_impl: using device CUDA0 (Tesla T4) (0000:00:04.0) - 14807 MiB free
llama_model_load_from_file_impl: using device CUDA1 (Tesla T4) (0000:00:05.0) - 14807 MiB free
llama_model_loader: loaded meta data with 38 key-value pairs and 579 tensors from /kaggle/working/models/Qwen2.5-14B-Instruct-Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 14B Instruct
llama_model_loader: - kv   3:     


Model successfully loaded on GPU.


In [7]:
def get_form_files(input_dir) -> List[Path]:
    files = sorted(input_dir.glob("*.txt"))
    print(f"Found {len(files)} files")
    return files

def load_text(file_path: Path) -> Tuple[str, str]:
    form_id = file_path.stem
    text = file_path.read_text(encoding='utf-8')
    return form_id, text

files = get_form_files(INPUT_DIR)
if files:
    test_form_id, test_text = load_text(files[0])
    print(f"\nExample: {test_form_id}")
    print(f"Text length: {len(test_text)} chars")
    print(f"part of text: {test_text[:1000]}")

Found 142 files

Example: 0000004457-23-000052
Text length: 136535 chars
part of text: About U HAUL HOLDING COMPANY...
>Item 1. Business
      




Company Overview






We are North America’s largest “do-it-yourself” moving and storage operator through our subsidiary U-Haul International, Inc. (“U-Haul”). U-Haul is synonymous with “do-it-yourself” moving and storage and is a leader in supplying products and services to help people move and store their household and commercial goods. Our primary service objective is to “provide a better and better product and service to more and more people at a lower and lower cost.” Unless the context otherwise requires, the terms “U-Haul Holding Company,” “Company,” “we,” “us,” or “our” refer to U-Haul Holding Company, a Nevada corporation, and all of its legal subsidiaries, on a consolidated basis. 






We were founded in 1945 as a sole proprietorship under the name "U-Haul Trailer Rental Company" and have rented trailers ever since. Starting in

In [8]:
SUMMARY_PROMPT = """Summarize the following SEC Form 10-K concisely. Focus on: business overview, risk factors, financial highlights. Keep under 300 words.\n\nText:\n{text}\n\nSummary:"""

def split_text(text: str, chunk_size: int = 20000, overlap: int = 1000) -> List[str]:
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

def summarize_single(text: str) -> str:
    prompt = SUMMARY_PROMPT.format(text=text[:25000])
    output = llm(
        prompt,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        echo=False
    )
    return output["choices"][0]["text"].strip()
    
def summarize_iterative(text: str) -> str:
    current_text = text
    iteration = 1
    while iteration <= 5:
        chunks = split_text(current_text)
        if len(chunks) == 1:
            return summarize_single(current_text)
        print(f"Iteration {iteration}: {len(chunks)} chunks")
        summaries = [summarize_single(chunk) for chunk in chunks]
        current_text = "\n\n".join(summaries)
        iteration += 1
    return summarize_single(current_text)

In [ ]:
from tqdm.auto import tqdm

def process_all_files(files: List[Path]) -> List[Dict[str, str]]:
    results = []
    for file_path in tqdm(files, desc="Summarizing"): 
        form_id, text = load_text(file_path)
        summary = summarize_iterative(text) 
        
        results.append({
            "formId": form_id,
            "summary": summary
        })
    
    return results

print(f"start processing {len(files)} files")
summaries = process_all_files(files)
print(f"summarized {len(summaries)}")

start processing 142 files


Summarizing:   0%|          | 0/142 [00:00<?, ?it/s]

ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture
ggml_cuda_graph_set_enabled: disabling CUDA graphs due to GPU architecture


Iteration 1: 8 chunks


llama_perf_context_print:        load time =    7458.09 ms
llama_perf_context_print: prompt eval time =    7456.92 ms /  4167 tokens (    1.79 ms per token,   558.81 tokens per second)
llama_perf_context_print:        eval time =  139159.48 ms /  2047 runs   (   67.98 ms per token,    14.71 tokens per second)
llama_perf_context_print:       total time =  151839.77 ms /  6214 tokens
llama_perf_context_print:    graphs reused =       2038
Llama.generate: 37 prefix-match hit, remaining 3683 prompt tokens to eval
llama_perf_context_print:        load time =    7458.09 ms
llama_perf_context_print: prompt eval time =    5616.42 ms /  3683 tokens (    1.52 ms per token,   655.76 tokens per second)
llama_perf_context_print:        eval time =  137041.68 ms /  2047 runs   (   66.95 ms per token,    14.94 tokens per second)
llama_perf_context_print:       total time =  148086.12 ms /  5730 tokens
llama_perf_context_print:    graphs reused =       2038
Llama.generate: 37 prefix-match hit, remaini

In [ ]:
import pandas as pd

df = pd.DataFrame(summaries)
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

from IPython.display import FileLink

display(FileLink("summaries.csv"))